# 🍃 PRANA ETL Pipeline — Tab 1: Ikhtisar Nasional (Executive Cockpit)

Notebook ini bertugas mengekstrak, membersihkan, menyelaraskan unit pengukuran, mengagregasi, serta mengekspor data dari **27 berkas AQMS telemetri**, **data surveilans rawat inap rumah sakit**, dan **tabel master geospasial** ke dalam satu berkas terstruktur:  
📁 **`dataset/dashboard/tab1_ikhtisar.json`**

---

### 🎯 Tujuan Analisis & Visualisasi Tab 1 (Multi-Dimensi Dinamis):
1. **Executive KPI Cockpit (4 Metrik Utama)**: Responsif terhadap rentang tanggal, polutan, dan opsi data SUSPECT.
   - **Rata-rata Konsentrasi Nasional**: Rata-rata polutan terpilih pada rentang tanggal aktif.
   - **Persentase Hari Melebihi Baku Mutu 24 Jam**: Tingkat pelanggaran PP 22/2021.
   - **Stasiun Paling Kritis**: Stasiun dengan beban konsentrasi tertinggi.
   - **Hospital Burden Index (%)**: Rasio estimasi beban admisi respirasi terhadap kapasitas kasur RS rujukan.
2. **Distribusi ISPU Dominan (Donut Chart)**: Proporsi stasiun berdasarkan kategori dominan (Permen LHK 14/2020).
3. **Peringkat Top 10 & Bottom 10 Stasiun**: Peringkat stasiun paling berpolusi vs terbersih.
4. **Peta Sebaran 27 Stasiun AQMS (Interactive Leaflet Map)**: Marker geospasial dengan kode warna ISPU dan radius pelanggaran 24 jam.
5. **Daily Multidimensional Cube**: Menyediakan deret harian 27 stasiun $\times$ 6 polutan $\times$ 2 status QC (`VALID` vs `VALID+SUSPECT`) berukuran $< 2\text{ MB}$ untuk memungkinkan seluruh filter bekerja seketika di peramban web.

---

### 📐 Standar Regulasi & Konversi yang Diterapkan:
- **PP No. 22 Tahun 2021** (Baku Mutu Udara Ambien 24 Jam & Tahunan):
  - $\text{PM}_{2.5}$: 24 jam $= 55\ \mu\text{g/m}^3$, tahunan $= 15\ \mu\text{g/m}^3$
  - $\text{PM}_{10}$: 24 jam $= 75\ \mu\text{g/m}^3$, tahunan $= 40\ \mu\text{g/m}^3$
  - $\text{SO}_2$: 24 jam $= 75\ \mu\text{g/m}^3$
  - $\text{CO}$: 8 jam $= 4.000\ \mu\text{g/m}^3$
  - $\text{O}_3$: 8 jam $= 100\ \mu\text{g/m}^3$
  - $\text{NO}_2$: 24 jam $= 65\ \mu\text{g/m}^3$
- **Permen LHK No. 14 Tahun 2020** (ISPU Breakpoints per Polutan)
- **Aturan Kelengkapan Data**: Minimal 75% jam valid per hari ($\ge 18$ jam dari 24 jam).

Normalisasi unit (12 November 2026): PM2.5/PM10 berlabel ppm pada BJB-01, MDN-01, dan PSW-01 diasumsikan sebagai skala massa logger mg/m³ dan dikalikan 1.000 ke µg/m³. Asumsi ini disetujui pengguna dan didukung pilihan satuan alat BAM, bukan bukti konversi ppm. Gas berlabel ppm pada tiga logger tersebut diasumsikan ppmv dan dikonversi dengan massa molekul/24,45 * 1.000 pada 25 C dan 1 atm (asumsi pengguna, bukan konfirmasi panitia). Kebijakan bersama berada di scripts/unit_normalization.py.


In [1]:
# =============================================================================
# LANGKAH 0: Inisialisasi Environment, Library, dan Path Direktori
# =============================================================================
import os, glob, json, time, sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# Definisi path proyek berbasis Pathlib
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if not (PROJECT_ROOT / 'dataset/raw/master/sites.csv').exists():
    PROJECT_ROOT = Path(r'D:/Projects/Gelarrasa')
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from unit_normalization import concentration_column, LOGGER_NORMALIZATION
AQMS_DIR       = PROJECT_ROOT / 'dataset/raw/aqms'
HEALTH_FILE    = PROJECT_ROOT / 'dataset/raw/health/HLT-01_health_outcomes_daily.csv'
SITES_FILE     = PROJECT_ROOT / 'dataset/raw/master/sites.csv'
HOSPITALS_FILE = PROJECT_ROOT / 'dataset/raw/master/hospitals.csv'
SENSORS_FILE   = PROJECT_ROOT / 'dataset/raw/master/sensors.csv'

OUT_DIR        = PROJECT_ROOT / 'dataset/dashboard'
OUT_FILE       = OUT_DIR / 'tab1_ikhtisar.json'
DASHBOARD_JSON = PROJECT_ROOT / 'dashboard/tab1_ikhtisar.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
aqms_files = sorted(list(AQMS_DIR.glob('*.csv')))
print('✅ Environment & Path siap!')
print(f'   Ditemukan {len(aqms_files)} berkas AQMS CSV di direktori: {AQMS_DIR}')

✅ Environment & Path siap!
   Ditemukan 27 berkas AQMS CSV di direktori: D:\GLRS\dataset\raw\aqms


In [2]:
# =============================================================================
# LANGKAH 1: Memuat Data Master (Sites, Sensors, & Hospitals)
# =============================================================================
sites     = pd.read_csv(SITES_FILE)
hospitals = pd.read_csv(HOSPITALS_FILE)
sensors   = pd.read_csv(SENSORS_FILE)

sensor_site = sensors[
    ['sensor_id', 'location_id', 'nama_sensor', 'model_sensor', 'status_operasional']
].merge(
    sites[[
        'location_id', 'station_code', 'kota_kabupaten', 'provinsi', 'kecamatan',
        'karakteristik_wilayah', 'latitude_dd', 'longitude_dd', 'region_group',
        'baseline_pm25', 'site_type'
    ]],
    on='location_id',
    how='left'
)
sensor_to_code = dict(zip(sensor_site['sensor_id'], sensor_site['station_code']))

print('✅ Data Master Berhasil Digabungkan:')
print(f'   Total Lokasi Stasiun : {len(sites)} stasiun')
print(f'   Total Sensor Terdaftar: {len(sensors)} sensor')
print(f'   Total Rumah Sakit     : {len(hospitals)} faskes rujukan')

✅ Data Master Berhasil Digabungkan:
   Total Lokasi Stasiun : 27 stasiun
   Total Sensor Terdaftar: 27 sensor
   Total Rumah Sakit     : 27 faskes rujukan


In [3]:
# =============================================================================
# LANGKAH 2: Membaca & Menyelaraskan (Normalize) 6 Kriteria Polutan dari 27 Berkas AQMS
# =============================================================================
# Asumsi logger partikulat mg/m³ ×1.000 hanya untuk BJB/MDN/PSW; gas ppmv memakai massa molekul/24,45 * 1.000 pada asumsi 25 C/1 atm.
# Membersihkan anomali fisik (nilai negatif dan infinity)

t_start = time.time()
pollutants = ['pm25', 'pm10', 'so2', 'co', 'o3', 'no2']
dfs = []
unavailable_by_station = {}
sites_by_code = {row['station_code']: row for row in sites.to_dict('records')}

for f in aqms_files:
    df = pd.read_csv(f, low_memory=False)
    station_code = sensor_to_code[df['sensor_id'].iloc[0]]
    unavailable_by_station[station_code] = []
    for p in pollutants:
        col_std = f'{p}_ug_m3'
        column = concentration_column(sites_by_code[station_code], p, df.columns)
        if column is None:
            df[col_std] = np.nan
            unavailable_by_station[station_code].append(p)
        else:
            source_column, factor = column
            df[col_std] = pd.to_numeric(df[source_column], errors='coerce') * factor

        # Pembersihan anomali tak terdefinisi dan non-fisik (negatif)
        df[col_std] = df[col_std].replace([np.inf, -np.inf], np.nan)
        df.loc[df[col_std] < 0, col_std] = np.nan
    
    cols_keep = ['sensor_id', 'timestamp_start', 'qc_flag'] + [f'{p}_ug_m3' for p in pollutants]
    dfs.append(df[cols_keep])

raw_aqms = pd.concat(dfs, ignore_index=True)
elapsed = time.time() - t_start
print(f'✅ Berhasil membaca & menstandarkan 27 berkas AQMS dalam {elapsed:.2f} detik!')
print(f'   Total Baris Data Mentah : {len(raw_aqms):,} observasi per jam')
print(f'   Sensor Terwakili        : {raw_aqms["sensor_id"].nunique()} dari 27 sensor')

✅ Berhasil membaca & menstandarkan 27 berkas AQMS dalam 1.34 detik!
   Total Baris Data Mentah : 667,981 observasi per jam
   Sensor Terwakili        : 27 dari 27 sensor


In [4]:
# =============================================================================
# LANGKAH 3: Parsing Waktu & Filter Kontrol Kualitas (QC Flag)
# =============================================================================
raw_aqms['ts'] = pd.to_datetime(raw_aqms['timestamp_start'], format='mixed', errors='coerce')
DATE_START = pd.Timestamp('2024-01-01')
DATE_END   = pd.Timestamp('2026-10-14 23:59:59')

in_window = raw_aqms[(raw_aqms['ts'] >= DATE_START) & (raw_aqms['ts'] <= DATE_END)].copy()
in_window['date'] = in_window['ts'].dt.strftime('%Y-%m-%d')
unique_dates = sorted(in_window['date'].unique())

valid_only = in_window[in_window['qc_flag'] == 'VALID'].copy()
valid_suspect = in_window[in_window['qc_flag'].isin(['VALID', 'SUSPECT'])].copy()

print(f'✅ Filter Kualitas Selesai:')
print(f'   Rentang Tanggal           : {unique_dates[0]} s.d. {unique_dates[-1]} ({len(unique_dates)} hari)')
print(f'   Total Baris VALID         : {len(valid_only):,} baris ({len(valid_only)/len(in_window)*100:.1f}%)')
print(f'   Total Baris VALID+SUSPECT : {len(valid_suspect):,} baris ({len(valid_suspect)/len(in_window)*100:.1f}%)')

✅ Filter Kualitas Selesai:
   Rentang Tanggal           : 2024-01-01 s.d. 2026-10-14 (1018 hari)
   Total Baris VALID         : 599,179 baris (89.7%)
   Total Baris VALID+SUSPECT : 618,901 baris (92.7%)


In [5]:
# =============================================================================
# LANGKAH 4: Agregasi Harian (24-Jam) dengan Aturan Kelengkapan >= 75% (18 Jam)
# =============================================================================
# Hari dengan jam valid < 18 ditandai NaN untuk menjaga integritas statistik

def aggregate_daily(df_subset):
    grouped = df_subset.groupby(['sensor_id', 'date'])
    counts = grouped['qc_flag'].count()
    means = grouped[[f'{p}_ug_m3' for p in pollutants]].mean()
    valid_mask = counts >= 18
    means[~valid_mask] = np.nan
    return means

daily_valid = aggregate_daily(valid_only)
daily_suspect = aggregate_daily(valid_suspect)

# Reindex ke matriks penuh (27 sensor x 1018 tanggal)
sensors_list = sorted(list(sensor_to_code.keys()))
full_idx = pd.MultiIndex.from_product([sensors_list, unique_dates], names=['sensor_id', 'date'])
df_v = daily_valid.reindex(full_idx)
df_s = daily_suspect.reindex(full_idx)

print('✅ Agregasi Harian Selesai:')
print(f'   Observasi Harian Valid         : {daily_valid.dropna(how="all").shape[0]:,} station-days')
print(f'   Observasi Harian Valid+Suspect : {daily_suspect.dropna(how="all").shape[0]:,} station-days')

✅ Agregasi Harian Selesai:
   Observasi Harian Valid         : 24,238 station-days
   Observasi Harian Valid+Suspect : 25,163 station-days


In [6]:
# =============================================================================
# LANGKAH 5: Membangun Daily Multidimensional Cube (Ringkas < 2 MB)
# =============================================================================
station_cube = {}
for sens in sensors_list:
    st_code = sensor_to_code[sens]
    st_v = df_v.loc[sens]
    st_s = df_s.loc[sens]
    station_cube[st_code] = {
        'unavailable': unavailable_by_station[st_code],
        'v': {
            p: [round(float(x), 1) if not np.isnan(x) else None for x in st_v[f'{p}_ug_m3']]
            for p in pollutants
        },
        's': {
            p: [round(float(x), 1) if not np.isnan(x) else None for x in st_s[f'{p}_ug_m3']]
            for p in pollutants
        }
    }

# Baku Mutu PP 22/2021 & Breakpoints ISPU Permen LHK 14/2020
thresholds = {
    'pm25': {'label': 'PM2.5', 'unit': 'µg/m³', 'pp22_24h': 55.0, 'pp22_annual': 15.0,
             'ispu_bins': [0.0, 15.5, 55.4, 150.4, 250.4],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
    'pm10': {'label': 'PM10', 'unit': 'µg/m³', 'pp22_24h': 75.0, 'pp22_annual': 40.0,
             'ispu_bins': [0.0, 50.0, 150.0, 350.0, 420.0],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
    'so2':  {'label': 'SO2', 'unit': 'µg/m³', 'pp22_24h': 75.0, 'pp22_annual': 40.0,
             'ispu_bins': [0.0, 52.0, 180.0, 400.0, 800.0],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
    'co':   {'label': 'CO', 'unit': 'µg/m³', 'pp22_24h': 4000.0, 'pp22_annual': 4000.0,
             'ispu_bins': [0.0, 4000.0, 8000.0, 15000.0, 30000.0],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
    'o3':   {'label': 'O3', 'unit': 'µg/m³', 'pp22_24h': 100.0, 'pp22_annual': 100.0,
             'ispu_bins': [0.0, 120.0, 235.0, 400.0, 800.0],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
    'no2':  {'label': 'NO2', 'unit': 'µg/m³', 'pp22_24h': 65.0, 'pp22_annual': 50.0,
             'ispu_bins': [0.0, 80.0, 200.0, 1130.0, 2260.0],
             'ispu_categories': ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']},
}
print(f'✅ Berhasil menyusun matriks kubus harian 27 stasiun!')

✅ Berhasil menyusun matriks kubus harian 27 stasiun!


In [7]:
# =============================================================================
# LANGKAH 6: Data Surveilans Kesehatan Harian Rumah Sakit (Nasional & Per Stasiun)
# =============================================================================
df_health = pd.read_csv(HEALTH_FILE)
df_health['date_dt'] = pd.to_datetime(df_health['date'], errors='coerce')
df_health = df_health[(df_health['date_dt'] >= DATE_START) & (df_health['date_dt'] <= DATE_END)].copy()
df_health['date_str'] = df_health['date_dt'].dt.strftime('%Y-%m-%d')
df_health['total_resp'] = pd.to_numeric(df_health['resp_admissions_under5'], errors='coerce').fillna(0) + \
                          pd.to_numeric(df_health['resp_admissions_over60'], errors='coerce').fillna(0)

daily_resp = df_health.groupby('date_str')['total_resp'].sum().reindex(unique_dates, fill_value=0.0)

# Pemetaan stasiun AQMS ke rumah sakit rujukan lokal (1-to-1 matching berdasarkan lokasi)
by_station_health = {}
for i in range(len(sites)):
    st_code = sites.iloc[i]['station_code']
    h_row = hospitals.iloc[i]
    h_id = h_row['hospital_id']
    beds = int(h_row['kapasitas_tempat_tidur'])
    
    sub = df_health[df_health['hospital_id'] == h_id]
    daily_s = sub.groupby('date_str')['total_resp'].sum().reindex(unique_dates, fill_value=0)
    
    by_station_health[st_code] = {
        'hospital_id': h_id,
        'hospital_name': str(h_row['nama_rumah_sakit']),
        'beds': beds,
        'admissions': [int(x) for x in daily_s.values]
    }

bed_capacity = float(pd.to_numeric(hospitals['kapasitas_tempat_tidur'], errors='coerce').fillna(0).sum())
health_daily = {
    'total_bed_capacity': int(bed_capacity),
    'total_hospitals': len(hospitals),
    'admissions_by_date': [round(float(x), 1) for x in daily_resp.values],
    'by_station': by_station_health
}
print('✅ Data Surveilans Kesehatan Harian (Nasional & Per Stasiun) Berhasil Dihubungkan!')


✅ Data Surveilans Kesehatan Harian (Nasional & Per Stasiun) Berhasil Dihubungkan!


In [8]:
# =============================================================================
# LANGKAH 7: Default Pre-Aggregation (PM2.5, VALID, Seluruh Periode)
# =============================================================================
# Menyediakan nilai siap saji agar dashboard dapat langsung merender tampilan awal seketika

ISPU_BINS   = [0.0, 15.0, 55.0, 150.0, 250.0, np.inf]
ISPU_LABELS = ['Baik', 'Sedang', 'Tidak Sehat', 'Sangat Tidak Sehat', 'Berbahaya']
ISPU_COLORS = {
    'Baik': '#10b981', 'Sedang': '#0284c7', 'Tidak Sehat': '#f59e0b',
    'Sangat Tidak Sehat': '#ef4444', 'Berbahaya': '#1e1b4b'
}

daily_pm25_raw = valid_only.groupby(['sensor_id', 'date'])['pm25_ug_m3'].mean().reset_index()
daily_pm25_raw['ispu_cat'] = pd.cut(daily_pm25_raw['pm25_ug_m3'], bins=ISPU_BINS, labels=ISPU_LABELS).astype(str)
daily_pm25_raw['exceed_24h'] = daily_pm25_raw['pm25_ug_m3'] > 55.0

station_stats = daily_pm25_raw.groupby('sensor_id').agg(
    n_days=('date', 'count'),
    n_exceed=('exceed_24h', 'sum'),
    pct_exceed_24h=('exceed_24h', lambda x: round(x.mean() * 100, 1)),
    avg_pm25=('pm25_ug_m3', lambda x: round(x.mean(), 2))
).reset_index()

dominant_ispu = daily_pm25_raw.groupby('sensor_id')['ispu_cat'].agg(lambda x: x.value_counts().idxmax()).reset_index().rename(columns={'ispu_cat': 'dominant_ispu_cat'})

# Data completeness
total_rows_per_sensor = in_window.groupby('sensor_id').size().reset_index(name='n_total_rows')
valid_rows_per_sensor = valid_only.groupby('sensor_id').size().reset_index(name='n_valid_rows')
completeness_df = total_rows_per_sensor.merge(valid_rows_per_sensor, on='sensor_id', how='left')
completeness_df['pct_valid'] = (completeness_df['n_valid_rows'] / completeness_df['n_total_rows'] * 100).round(1)

station_df = (
    station_stats
    .merge(dominant_ispu, on='sensor_id', how='left')
    .merge(completeness_df[['sensor_id', 'pct_valid']], on='sensor_id', how='left')
    .merge(sensor_site, on='sensor_id', how='left')
)
station_df['ispu_color'] = station_df['dominant_ispu_cat'].map(ISPU_COLORS)
station_df = station_df.sort_values('avg_pm25', ascending=False).reset_index(drop=True)
station_df['rank'] = range(1, len(station_df) + 1)

top_critical = station_df.iloc[0]
avg_pm25_nat = round(float(station_df['avg_pm25'].mean()), 2)
pct_exceed_nat = round(float(daily_pm25_raw['exceed_24h'].mean() * 100), 1)
avg_daily_adm = daily_resp.mean()
burden_idx = round(float((avg_daily_adm / bed_capacity) * 100), 2)

print('===========================================================')
print('     🏆 4 KPI UTAMA DEFAULT (PM2.5 • VALID • 2024-2026)   ')
print('===========================================================')
print(f'1. Rata-Rata PM2.5 Nasional        : {avg_pm25_nat} µg/m³')
print(f'2. % Hari Melebihi Batas PP22/2021 : {pct_exceed_nat}% hari')
print(f'3. Stasiun Paling Kritis           : {top_critical["station_code"]} ({top_critical["kota_kabupaten"]}) [{top_critical["avg_pm25"]} µg/m³]')
print(f'4. Hospital Burden Index           : {burden_idx}%')
print('===========================================================')

     🏆 4 KPI UTAMA DEFAULT (PM2.5 • VALID • 2024-2026)   
1. Rata-Rata PM2.5 Nasional        : 43.25 µg/m³
2. % Hari Melebihi Batas PP22/2021 : 17.7% hari
3. Stasiun Paling Kritis           : AQMS-BJB-01 (Kota Banjarbaru) [102.14 µg/m³]
4. Hospital Burden Index           : 6.43%


In [9]:
# =============================================================================
# LANGKAH 8: Data Donut Chart, Ranking, dan Marker Peta Geospasial
# =============================================================================
map_stations = []
for _, row in station_df.iterrows():
    map_stations.append({
        'station_code': str(row['station_code']),
        'station_name': str(row['nama_sensor']),
        'kota': str(row['kota_kabupaten']),
        'provinsi': str(row['provinsi']),
        'region_group': str(row['region_group']),
        'karakteristik': str(row['karakteristik_wilayah']),
        'site_type': str(row['site_type']),
        'lat': float(row['latitude_dd']),
        'lon': float(row['longitude_dd']),
        'avg_pm25': float(row['avg_pm25']),
        'dominant_ispu_cat': str(row['dominant_ispu_cat']),
        'ispu_color': str(row['ispu_color']),
        'pct_exceed_24h': float(row['pct_exceed_24h']),
        'pct_valid': float(row['pct_valid']),
        'rank': int(row['rank']),
    })

donut_counts = station_df['dominant_ispu_cat'].value_counts()
ispu_donut_data = []
for cat in ISPU_LABELS:
    cnt = int(donut_counts.get(cat, 0))
    if cnt > 0:
        ispu_donut_data.append({
            'category': cat,
            'count': cnt,
            'percentage': round(cnt / len(station_df) * 100, 1),
            'color': ISPU_COLORS.get(cat, '#94a3b8')
        })

print(f'✅ Berhasil menyiapkan {len(map_stations)} marker peta & donat distribusi!')

✅ Berhasil menyiapkan 27 marker peta & donat distribusi!


In [10]:
# =============================================================================
# LANGKAH 9: Ekspor Berkas JSON Final (Dioptimalkan < 2 MB)
# =============================================================================
final_output = {
    'meta': {
        'dashboard_title': 'National Executive Cockpit • Tab 1',
        'team_name': 'TIM NASI BAKAR CUMI HITAM PAK KRIS',
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'date_range_start': unique_dates[0],
        'date_range_end': unique_dates[-1],
        'total_stations': len(station_df),
        'unit_normalization': LOGGER_NORMALIZATION,
        'total_valid_records': int(len(valid_only)),
        'pp22_threshold_24h_ugm3': 55.0,
        'pp22_threshold_annual_ugm3': 15.0,
        'who_guideline_annual_ugm3': 5.0,
    },
    'kpi': {
        'avg_pm25_national': avg_pm25_nat,
        'pct_days_exceed_24h': pct_exceed_nat,
        'station_most_critical': {
            'station_code': str(top_critical['station_code']),
            'station_name': str(top_critical['nama_sensor']),
            'kota': str(top_critical['kota_kabupaten']),
            'provinsi': str(top_critical['provinsi']),
            'avg_pm25': float(top_critical['avg_pm25']),
            'pct_exceed': float(top_critical['pct_exceed_24h']),
        },
        'hospital_burden_index': burden_idx,
        'hospital_burden_detail': {
            'avg_daily_resp_admissions': round(float(avg_daily_adm), 1),
            'total_bed_capacity': int(bed_capacity),
            'total_hospitals': len(hospitals),
            'days_observed': len(unique_dates),
        }
    },
    'ispu_donut': ispu_donut_data,
    'pm25_ranking': map_stations,
    'map_stations': map_stations,
    'daily_cube': {
        'dates': unique_dates,
        'pollutants': pollutants,
        'thresholds': thresholds,
        'stations': station_cube,
        'health_daily': health_daily
    }
}

raw_str = json.dumps(final_output, ensure_ascii=False, separators=(',', ':'))
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write(raw_str)
with open(DASHBOARD_JSON, 'w', encoding='utf-8') as f:
    f.write(raw_str)

size_mb = len(raw_str.encode('utf-8')) / (1024 * 1024)
print('===========================================================')
print('🎉 SUKSES! File JSON Tab 1 Multi-Dimensi Berhasil Disimpan!')
print(f'   Lokasi Output 1 : {OUT_FILE}')
print(f'   Lokasi Output 2 : {DASHBOARD_JSON}')
print(f'   Ukuran File     : {size_mb:.2f} MB (< 2 MB Sesuai Spesifikasi)')
print('===========================================================')

🎉 SUKSES! File JSON Tab 1 Multi-Dimensi Berhasil Disimpan!
   Lokasi Output 1 : D:\GLRS\dataset\dashboard\tab1_ikhtisar.json
   Lokasi Output 2 : D:\GLRS\dashboard\tab1_ikhtisar.json
   Ukuran File     : 1.72 MB (< 2 MB Sesuai Spesifikasi)


In [11]:
# =============================================================================
# LANGKAH 10: Sanity Check & Verifikasi Struktur Payload
# =============================================================================
with open(OUT_FILE, 'r', encoding='utf-8') as f:
    verify = json.load(f)

cube = verify['daily_cube']
print('✅ Verifikasi JSON:')
print(f"   Judul Metadata       : {verify['meta']['dashboard_title']}")
print(f"   Rentang Waktu        : {verify['meta']['date_range_start']} s.d. {verify['meta']['date_range_end']}")
print(f"   Jumlah Tanggal Cube  : {len(cube['dates'])} hari")
print(f"   Daftar Polutan Cube  : {cube['pollutants']}")
print(f"   Jumlah Stasiun Cube  : {len(cube['stations'])}")
print(f"   Contoh Stasiun       : {list(cube['stations'].keys())[0]}")
print(f"   Data Surveilans RS   : {len(cube['health_daily']['admissions_by_date'])} hari tercatat")
print('\nPayload siap digunakan secara dinamis oleh seluruh filter frontend!')

✅ Verifikasi JSON:
   Judul Metadata       : National Executive Cockpit • Tab 1
   Rentang Waktu        : 2024-01-01 s.d. 2026-10-14
   Jumlah Tanggal Cube  : 1018 hari
   Daftar Polutan Cube  : ['pm25', 'pm10', 'so2', 'co', 'o3', 'no2']
   Jumlah Stasiun Cube  : 27
   Contoh Stasiun       : AQMS-JKT-01
   Data Surveilans RS   : 1018 hari tercatat

Payload siap digunakan secara dinamis oleh seluruh filter frontend!
